# Pandas Tutorial — Lecture 4

This notebook is a **student-friendly tutorial** that matches the topics covered in **Lecture 4: Pandas**.

**What you’ll learn**
- What pandas is and why it’s useful
- Series vs DataFrame
- Loading/saving files (CSV, TXT, JSON)
- Inspecting data (`head`, `info`, `describe`, …)
- Selecting data (`[]`, `.loc`, `.iloc`) and the “inclusive vs exclusive” slicing rule
- Cleaning & manipulating (add/drop/rename columns, missing values, sorting)
- Aggregation & grouping (`groupby`)
- Combining DataFrames (`concat`, `merge`)
- Practice exercises (with expected outputs / hints)

---

> Tip: Run each cell top-to-bottom. Don’t skip the exercises 🙂


## 0) Setup

In Colab, most libraries are already installed. If you ever need to install pandas, use `!pip install pandas`.

We’ll import pandas (and NumPy for a few examples).


In [ ]:
import pandas as pd
import numpy as np

pd.__version__


## 1) What is Pandas?

**pandas** is a Python library for **data manipulation and analysis**.  
It builds on top of **NumPy** and adds powerful, easy-to-use **tabular data structures**.

Key idea:
- **Series** = 1D labeled array (like a single column)
- **DataFrame** = 2D labeled table (like a spreadsheet / SQL table)


## 2) Series (1D)

A **Series** holds:
- values (data)
- an **index** (labels)

When created from a list, pandas creates a default integer index starting at 0.


In [ ]:
# Series from a list
s_list = pd.Series([10, 20, 30, 40, 50])
s_list


### Series from a dictionary

When you create a Series from a dictionary, the dictionary **keys become the index**.


In [ ]:
data = {"Math": 90, "Science": 85, "English": 92}
s_dict = pd.Series(data)
s_dict


## 3) DataFrame (2D)

A **DataFrame** has:
- rows (index)
- columns (labels)
- each column can have a different type (numbers, text, dates, ...)


In [ ]:
# DataFrame from a dictionary of lists
data = {
    "Student": ["Alice", "Bob", "Charlie"],
    "Math": [90, 85, 92],
    "Science": [88, 91, 80],
}
df_students = pd.DataFrame(data)
df_students


### DataFrame from a list of dictionaries (JSON-like)

Useful when you have records like API responses or JSON files.


In [ ]:
records = [
    {"Name": "Alice", "Age": 24},
    {"Name": "Bob", "Age": 27},
]
df_records = pd.DataFrame(records)
df_records


## 4) Loading a dataset (Titanic)

There are 2 easy options in Colab:
1) **Seaborn built-in** dataset (no file needed)
2) Load a CSV from a URL

We’ll use a URL so you practice `read_csv`.


In [ ]:
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

df.head()


## 5) Initial data inspection

Common commands:
- `df.head(n)` / `df.tail(n)` / `df.sample(n)`
- `df.info()` (types + missing values)
- `df.describe()` (numeric summary)
- `df.shape`, `df.columns`


In [ ]:
df.head(3)


In [ ]:
df.tail(3)


In [ ]:
df.sample(2, random_state=42)


In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
df.shape, df.columns


## 6) Selecting columns

- Single column: `df['Age']` → returns a **Series**
- Multiple columns: `df[['Name','Age']]` → returns a **DataFrame** (note the double brackets)


In [ ]:
age = df["Age"]
type(age), age.head()


In [ ]:
subset = df[["Name", "Age", "Sex", "Survived"]]
subset.head()


## 7) `.loc` vs `.iloc` (and “last index inclusiveness”)

### `.loc` (label-based)
- Uses **labels** (row labels / column names)
- Slicing is **inclusive** of the end label

### `.iloc` (position-based)
- Uses **integer positions** (0-based)
- Slicing is like Python: **end is exclusive**

Let’s see it.


In [ ]:
# Using loc with labels (inclusive end)
df.loc[0:2, ["Name", "Age", "Survived"]]


In [ ]:
# Using iloc with positions (exclusive end)
df.iloc[0:3, [2, 5, 1]]  # columns by position: Name, Age, Survived (positions vary by dataset)


> If you’re not sure about column positions, use `df.columns` to check, or just use `.loc` with column names.


## 8) Conditional filtering

Example: passengers older than 60.


In [ ]:
df[df["Age"] > 60][["Name", "Age", "Survived"]].head(10)


## 9) Manipulating columns (add / drop)

### Add a new column
Example: create `FamilySize` using `SibSp` + `Parch` (siblings/spouses + parents/children).

### Drop columns
Use `df.drop(..., axis=1)` to drop columns.


In [ ]:
df2 = df.copy()
df2["FamilySize"] = df2["SibSp"] + df2["Parch"]
df2[["SibSp", "Parch", "FamilySize"]].head()


In [ ]:
# Drop 1 column (returns a NEW DataFrame because inplace=False by default)
df3 = df2.drop("Cabin", axis=1)
df3.columns


**Remember**
- `axis=1` targets **columns**
- `axis=0` targets **rows**
- `inplace=True` modifies the original DataFrame (often avoided; many prefer `df = df.drop(...)`)


## 10) Renaming columns

Use `.rename(columns={old:new})`.


In [ ]:
df4 = df3.rename(columns={"Pclass": "PassengerClass", "Sex": "Gender"})
df4[["PassengerClass", "Gender"]].head()


## 11) Missing values (NaN)

Steps:
1) Find missing data: `df.isnull().sum()`
2) Drop missing rows/columns: `dropna`
3) Fill missing values (imputation): `fillna`

We’ll fill missing `Age` using the mean, then drop rows missing `Embarked`.


In [ ]:
df4.isnull().sum().sort_values(ascending=False).head(10)


In [ ]:
df5 = df4.copy()
df5["Age"] = df5["Age"].fillna(df5["Age"].mean())
df5 = df5.dropna(subset=["Embarked"])

df5.isnull().sum().sort_values(ascending=False).head(10)


## 12) Sorting

- `sort_values(by="Age")`
- Multiple columns: `sort_values(by=["PassengerClass","Fare"], ascending=[True, False])`


In [ ]:
df5.sort_values(by="Age", ascending=True)[["Name", "Age", "Fare"]].head(10)


In [ ]:
df5.sort_values(by=["PassengerClass", "Fare"], ascending=[True, False])[["PassengerClass", "Name", "Fare"]].head(10)


## 13) Aggregation (basic stats)

Common methods:
- `.mean()`, `.median()`, `.sum()`, `.count()`, `.min()`, `.max()`
- `.value_counts()`


In [ ]:
df5["Fare"].mean(), df5["Fare"].median(), df5["Fare"].max()


In [ ]:
df5["Embarked"].value_counts()


## 14) GroupBy (Split–Apply–Combine)

Example: mean survival rate by passenger class.


In [ ]:
df5.groupby("PassengerClass")["Survived"].mean().sort_index()


### Group by multiple columns

Example: survival rate by class and gender.


In [ ]:
df5.groupby(["PassengerClass", "Gender"])["Survived"].mean()


## 15) Combining DataFrames

### `pd.concat`
- `axis=0`: stack rows (one on top of another)
- `axis=1`: stack columns side-by-side (align on index)

### `pd.merge`
SQL-style joins on a key (like `StudentID`).


In [ ]:
df_a = pd.DataFrame({"id":[1,2,3], "score":[80, 90, 75]})
df_b = pd.DataFrame({"id":[4,5], "score":[88, 92]})

pd.concat([df_a, df_b], axis=0, ignore_index=True)


In [ ]:
students = pd.DataFrame({
    "StudentID":[1,2,3],
    "Name":["Alice","Bob","Charlie"]
})
courses = pd.DataFrame({
    "StudentID":[1,1,3],
    "Course":["Math","Science","History"]
})

pd.merge(students, courses, on="StudentID", how="inner")


## 16) Saving data (CSV / TXT / JSON)

### CSV
- `index=False` usually avoids exporting the extra index column.

### TXT (space-separated)
- Use `sep=' '`.

### JSON
- `orient="records"` is a very common, clean layout (list of row objects).


In [ ]:
# We'll save a small sample so you don't download a huge file by accident.
sample = df5[["Name", "Age", "Fare", "Survived", "PassengerClass", "Gender"]].head(20)

sample.to_csv("diabetes_out.csv", index=False)
sample.to_csv("diabetes_out.txt", index=False, sep=" ")
sample.to_json("diabetes_out.json", orient="records", indent=2)

# Show files created in Colab
import os
[f for f in os.listdir(".") if f.startswith("diabetes_out")]


# ✅ Practice Exercises

---



Try these exercises. Write your code in the empty cells and run them.

### Exercise 1
Load the Titanic dataset again into `df_ex` and print:
- `df_ex.shape`
- first 5 rows

### Exercise 2
Using `.loc`, select rows **0 to 3** (inclusive) and columns `Name`, `Age`, `Survived`.

### Exercise 3
Using `.iloc`, select the **first 4 rows** and the **first 3 columns**.

### Exercise 4
Create a new column `IsChild` where `Age < 18`.

### Exercise 5
Compute the survival rate (mean of `Survived`) by `Sex`.

### Exercise 6 (Challenge)
Find the **top 10 highest fares** among passengers with:
- `PassengerClass == 1`
- `Age` not missing
Show `Name`, `Age`, `Fare`.

---
**Hints**
- Use `df.isnull().sum()` to check missing values.
- Use `sort_values(by=..., ascending=...)` for ranking.


In [ ]:
# Exercise 1 (your code)
import pandas as pd
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df_ex = pd.read_csv(url)
print(df_ex.shape)
df_ex.head()

(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
# Exercise 2 (your code)
df_ex.loc[0:3, ["Name", "Age", "Survived"]]

,Name,Age,Survived
0,"Braund, Mr. Owen Harris",22.0,0
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1
2,"Heikkinen, Miss. Laina",26.0,1
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1


In [ ]:
# Exercise 3 (your code)
df_ex.iloc[0:4, 0:3]

,PassengerId,Survived,Pclass
0,1,0,3
1,2,1,1
2,3,1,3
3,4,1,1


In [ ]:
# Exercise 4 (your code)
df_ex["IsChild"] = df_ex["Age"] < 18

In [ ]:
# Exercise 5 (your code)
df_ex.groupby("Sex")["Survived"].mean()

,Survived
Sex,
female,0.742038
male,0.188908


In [ ]:
# Exercise 6 (your code)
mask = (df_ex["Pclass"] == 1) & (df_ex["Age"].notnull())
df_ex[mask].sort_values(by="Fare", ascending=False).head(10)[["Name", "Age", "Fare"]]

,Name,Age,Fare
258,"Ward, Miss. Anna",35.0,512.3292
737,"Lesurer, Mr. Gustave J",35.0,512.3292
679,"Cardeza, Mr. Thomas Drake Martinez",36.0,512.3292
341,"Fortune, Miss. Alice Elizabeth",24.0,263.0000
88,"Fortune, Miss. Mabel Helen",23.0,263.0000
27,"Fortune, Mr. Charles Alexander",19.0,263.0000
438,"Fortune, Mr. Mark",64.0,263.0000
311,"Ryerson, Miss. Emily Borie",18.0,262.3750
742,"Ryerson, Miss. Susan Parker ""Suzette""",21.0,262.3750
118,"Baxter, Mr. Quigg Edmond",24.0,247.5208
